# Export Base Template — DuckDB Workflow

**Template ID:** EXPORT-BASE  
**Workflow layer:** `curated → output`  
**Primary mode:** Notebook + SQL via DuckDB Python API

## Purpose

Notebook-first export and delivery template for DuckDB pipelines. Select a curated table or query, export CSV and Parquet (including partitioned layouts and GeoParquet), build a validation summary, write a manifest, and assemble a stakeholder-ready delivery package.

Follows the layered workflow convention:

```text
source → raw → staging → curated → output
```

Delivery package layout:

```text
output/
├── data/
├── validation/
├── README.md
└── manifest.csv
```

## How to use

1. Copy this notebook into your project (or run it from this repo).
2. Run validation first (`notebooks/03_validation_base.ipynb`) — export only after checks pass.
3. Update **Project configuration** with table names, output paths, and partition columns.
4. Run all cells top to bottom.
5. Review the **Final delivery checklist** before publishing the package.

**Example table:** `curated.fct_orders` (TPC-H `lineitem` model from `notebooks/01_etl_base.ipynb`)  
**Target users:** analysts, data engineers, GIS users, SQL users, Python users.

## 2. Project configuration

Edit the variables below for your delivery. Set `EXPORT_QUERY = None` to export the full `SOURCE_TABLE_NAME`; otherwise supply a SQL string for a filtered or projected export.

| Variable | Purpose |
|----------|----------|
| `SOURCE_TABLE_NAME` | Curated table to export (e.g. `curated.fct_orders`) |
| `OUTPUT_DIR` | Base folder for standalone exports under `data/output/` |
| `CSV_OUTPUT_PATH` | Destination for flat CSV export |
| `PARQUET_OUTPUT_PATH` | Destination for single-file Parquet export |
| `PARTITION_COLUMNS` | Hive-style partition keys; empty list skips partitioned export |
| `GEOPARQUET_OUTPUT_PATH` | GeoParquet destination when spatial export is enabled |
| `DELIVERY_PACKAGE_DIR` | Root of the handoff folder (`data/`, `validation/`, `README.md`, `manifest.csv`) |

See `docs/10_export/` for format-specific guidance.

In [ ]:
from datetime import date
from pathlib import Path

# --- paths ---
PROJECT_ROOT = Path("..").resolve()  # repo root when notebook lives in notebooks/
DB_PATH = PROJECT_ROOT / "work.duckdb"

# --- source table or query ---
SOURCE_TABLE_NAME = "curated.fct_orders"
EXPORT_QUERY = None  # e.g. "SELECT * FROM curated.fct_orders WHERE order_status = 'fulfilled'"

# --- standalone export paths ---
OUTPUT_DIR = PROJECT_ROOT / "data" / "output"
CSV_OUTPUT_PATH = OUTPUT_DIR / "fct_orders.csv"
PARQUET_OUTPUT_PATH = OUTPUT_DIR / "fct_orders.parquet"
PARTITIONED_PARQUET_DIR = OUTPUT_DIR / "fct_orders_partitioned"
PARTITION_COLUMNS = ["order_month"]  # [] to skip partitioned export

# --- spatial export (optional) ---
GEOMETRY_COLUMN = None  # e.g. "geom"; set to enable GeoParquet section
GEOPARQUET_OUTPUT_PATH = OUTPUT_DIR / "geo_parcels.parquet"
EXPORT_GEOPARQUET = GEOMETRY_COLUMN is not None

# --- delivery package ---
DELIVERY_DATE = date.today().isoformat()
DELIVERY_PACKAGE_DIR = PROJECT_ROOT / "data" / "output" / f"delivery_{DELIVERY_DATE}"
DELIVERY_DATA_DIR = DELIVERY_PACKAGE_DIR / "data"
DELIVERY_VALIDATION_DIR = DELIVERY_PACKAGE_DIR / "validation"
DELIVERY_MANIFEST_PATH = DELIVERY_PACKAGE_DIR / "manifest.csv"
DELIVERY_README_PATH = DELIVERY_PACKAGE_DIR / "README.md"

# --- export toggles ---
EXPORT_CSV = True
EXPORT_PARQUET = True
EXPORT_PARTITIONED_PARQUET = bool(PARTITION_COLUMNS)
COPY_EXPORTS_TO_PACKAGE = True  # copy standalone exports into DELIVERY_PACKAGE_DIR/data/

# --- validation summary columns (non-spatial defaults) ---
MEASURE_COLUMN = "amount"  # numeric column for sum check; None to skip
STATUS_COLUMN = "order_status"  # categorical column for distribution check; None to skip

# --- bootstrap (runnable demo without prior ETL notebook) ---
BOOTSTRAP_EXAMPLE_TABLE = True  # set False when curated table already exists
RAW_INPUT_PATH = "https://shell.duckdb.org/data/tpch/0_01/parquet/lineitem.parquet"
RAW_ROW_LIMIT = 50_000

## 3. Imports

In [ ]:
import shutil

import duckdb
import pandas as pd
from IPython.display import display

## 4. Connect to DuckDB

Opens (or creates) a persistent database and ensures workflow schemas exist.

In [ ]:
con = duckdb.connect(str(DB_PATH))

for schema in ("raw", "staging", "curated"):
    con.execute(f'CREATE SCHEMA IF NOT EXISTS "{schema}";')

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DELIVERY_DATA_DIR.mkdir(parents=True, exist_ok=True)
DELIVERY_VALIDATION_DIR.mkdir(parents=True, exist_ok=True)

print(f"Connected to: {DB_PATH}")
display(con.sql("SELECT version() AS duckdb_version").df())

## 5. Select table or query to export

Preview the export source. Optionally bootstrap the example `curated.fct_orders` table from public TPC-H Parquet when running standalone.

In [ ]:
if BOOTSTRAP_EXAMPLE_TABLE:
    con.execute("INSTALL httpfs; LOAD httpfs;")

    bootstrap_sql = f"""
    CREATE OR REPLACE TABLE {SOURCE_TABLE_NAME} AS
    WITH lineitem AS (
      SELECT * FROM read_parquet('{RAW_INPUT_PATH}')
      {"LIMIT " + str(RAW_ROW_LIMIT) if RAW_ROW_LIMIT else ""}
    )
    SELECT
      l_orderkey AS order_id,
      l_linenumber AS line_number,
      l_suppkey AS supplier_id,
      CAST(l_shipdate AS DATE) AS order_date,
      DATE_TRUNC('month', CAST(l_shipdate AS DATE)) AS order_month,
      l_extendedprice AS amount,
      l_quantity AS quantity,
      CASE l_returnflag
        WHEN 'R' THEN 'returned'
        WHEN 'F' THEN 'fulfilled'
        ELSE 'open'
      END AS order_status,
      l_shipmode AS ship_mode
    FROM lineitem
    WHERE l_returnflag <> 'R'
      AND l_extendedprice IS NOT NULL
      AND l_extendedprice > 0;
    """
    con.execute(bootstrap_sql)
    print(f"Bootstrapped example table → {SOURCE_TABLE_NAME}")

export_source_sql = EXPORT_QUERY if EXPORT_QUERY else f"SELECT * FROM {SOURCE_TABLE_NAME}"

preview_sql = f"""
SELECT *
FROM ({export_source_sql}) AS export_src
LIMIT 10;
"""

count_sql = f"""
SELECT COUNT(*) AS export_row_count
FROM ({export_source_sql}) AS export_src;
"""

export_row_count = con.sql(count_sql).df().export_row_count.iloc[0]
print(f"Export source: {SOURCE_TABLE_NAME if not EXPORT_QUERY else 'custom query'}")
print(f"Rows to export: {export_row_count:,}")
display(con.sql(preview_sql).df())

## 6. Export CSV

Write a spreadsheet-friendly CSV. See `docs/10_export/csv_export.md` and `docs/10_export/excel_ready_csv.md`.

In [ ]:
if EXPORT_CSV:
    csv_path = CSV_OUTPUT_PATH.resolve().as_posix()
    CSV_OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

    csv_sql = f"""
    COPY (
      {export_source_sql}
    ) TO '{csv_path}'
    (HEADER, DELIMITER ',');
    """
    con.execute(csv_sql)

    verify_sql = f"""
    SELECT COUNT(*) AS exported_row_count
    FROM read_csv('{csv_path}', header = true);
    """
    csv_rows = con.sql(verify_sql).df().exported_row_count.iloc[0]

    print(f"Exported CSV → {CSV_OUTPUT_PATH}")
    print(f"Table rows: {export_row_count:,} | File rows: {csv_rows:,}")
    assert csv_rows == export_row_count, "CSV row count mismatch"
else:
    print("Skipping CSV export (EXPORT_CSV = False).")

## 7. Export Parquet

Write a single ZSTD-compressed Parquet file — primary analytics format. See `docs/10_export/parquet_export.md`.

In [ ]:
if EXPORT_PARQUET:
    parquet_path = PARQUET_OUTPUT_PATH.resolve().as_posix()
    PARQUET_OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

    parquet_sql = f"""
    COPY (
      {export_source_sql}
    ) TO '{parquet_path}'
    (FORMAT PARQUET, COMPRESSION ZSTD);
    """
    con.execute(parquet_sql)

    verify_sql = f"""
    SELECT COUNT(*) AS exported_row_count
    FROM read_parquet('{parquet_path}');
    """
    parquet_rows = con.sql(verify_sql).df().exported_row_count.iloc[0]

    print(f"Exported Parquet → {PARQUET_OUTPUT_PATH}")
    print(f"Table rows: {export_row_count:,} | File rows: {parquet_rows:,}")
    assert parquet_rows == export_row_count, "Parquet row count mismatch"
else:
    print("Skipping Parquet export (EXPORT_PARQUET = False).")

## 8. Export partitioned Parquet

Write hive-style partitioned Parquet when `PARTITION_COLUMNS` is non-empty. See `docs/10_export/partitioned_parquet_export.md`.

In [ ]:
if EXPORT_PARTITIONED_PARQUET and PARTITION_COLUMNS:
    partitioned_dir = PARTITIONED_PARQUET_DIR.resolve().as_posix()
    if PARTITIONED_PARQUET_DIR.exists():
        shutil.rmtree(PARTITIONED_PARQUET_DIR)
    PARTITIONED_PARQUET_DIR.mkdir(parents=True, exist_ok=True)

    partition_clause = ", ".join(PARTITION_COLUMNS)
    partitioned_sql = f"""
    COPY (
      {export_source_sql}
    ) TO '{partitioned_dir}'
    (FORMAT PARQUET, PARTITION_BY ({partition_clause}), COMPRESSION ZSTD);
    """
    con.execute(partitioned_sql)

    verify_sql = f"""
    SELECT COUNT(*) AS exported_row_count
    FROM read_parquet('{partitioned_dir}/*/*.parquet', hive_partitioning = true);
    """
    part_rows = con.sql(verify_sql).df().exported_row_count.iloc[0]

    print(f"Exported partitioned Parquet → {PARTITIONED_PARQUET_DIR}")
    print(f"Partition keys: {PARTITION_COLUMNS}")
    print(f"Table rows: {export_row_count:,} | Partitioned file rows: {part_rows:,}")
    assert part_rows == export_row_count, "Partitioned Parquet row count mismatch"
else:
    print(
        "Skipping partitioned Parquet export. "
        "Set PARTITION_COLUMNS (e.g. ['order_month']) to enable."
    )

## 9. Export GeoParquet if spatial

Write geometry to GeoParquet via the spatial extension when `GEOMETRY_COLUMN` is set. See `docs/10_export/geoparquet_export.md`.

In [ ]:
if EXPORT_GEOPARQUET and GEOMETRY_COLUMN:
    con.execute("INSTALL spatial; LOAD spatial;")

    geoparquet_path = GEOPARQUET_OUTPUT_PATH.resolve().as_posix()
    GEOPARQUET_OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

    geoparquet_sql = f"""
    COPY (
      {export_source_sql}
    ) TO '{geoparquet_path}'
    (FORMAT GDAL, DRIVER 'Parquet');
    """
    con.execute(geoparquet_sql)

    file_rows = con.sql(
        f"SELECT COUNT(*) AS row_count FROM ST_Read('{geoparquet_path}')"
    ).df().row_count.iloc[0]

    invalid_sql = f"""
    SELECT SUM(CASE WHEN NOT ST_IsValid({GEOMETRY_COLUMN}) THEN 1 ELSE 0 END) AS invalid_count
    FROM ST_Read('{geoparquet_path}');
    """
    invalid_count = con.sql(invalid_sql).df().invalid_count.iloc[0]

    print(f"Exported GeoParquet → {GEOPARQUET_OUTPUT_PATH}")
    print(f"Table rows: {export_row_count:,} | File rows: {file_rows:,}")
    print(f"Invalid geometries in export: {invalid_count}")
else:
    print(
        f"Skipping GeoParquet export. Set GEOMETRY_COLUMN (e.g. 'geom') to write "
        f"{GEOPARQUET_OUTPUT_PATH}"
    )

## 10. Create validation summary

Write export QA metrics to `validation/summary.csv` inside the delivery package. Extend checks for spatial layers (valid geometry, extent) as needed.

In [ ]:
validation_summary_path = (DELIVERY_VALIDATION_DIR / "summary.csv").resolve().as_posix()

checks = [
    f"""
    SELECT
      '{SOURCE_TABLE_NAME}' AS dataset,
      'row_count' AS check_name,
      CAST(COUNT(*) AS VARCHAR) AS result_value,
      'PASS' AS status
    FROM ({export_source_sql}) AS src
    """
]

if MEASURE_COLUMN:
    checks.append(f"""
    SELECT
      '{SOURCE_TABLE_NAME}',
      'sum_{MEASURE_COLUMN}',
      CAST(ROUND(SUM({MEASURE_COLUMN}), 2) AS VARCHAR),
      CASE WHEN SUM({MEASURE_COLUMN}) > 0 THEN 'PASS' ELSE 'FAIL' END
    FROM ({export_source_sql}) AS src
    """)

if STATUS_COLUMN:
    checks.append(f"""
    SELECT
      '{SOURCE_TABLE_NAME}',
      'distinct_{STATUS_COLUMN}',
      CAST(COUNT(DISTINCT {STATUS_COLUMN}) AS VARCHAR),
      'PASS'
    FROM ({export_source_sql}) AS src
    """)

if EXPORT_GEOPARQUET and GEOMETRY_COLUMN:
    checks.append(f"""
    SELECT
      '{SOURCE_TABLE_NAME}',
      'invalid_geom',
      CAST(SUM(CASE WHEN NOT ST_IsValid({GEOMETRY_COLUMN}) THEN 1 ELSE 0 END) AS VARCHAR),
      CASE
        WHEN SUM(CASE WHEN NOT ST_IsValid({GEOMETRY_COLUMN}) THEN 1 ELSE 0 END) = 0
        THEN 'PASS' ELSE 'FAIL'
      END
    FROM ({export_source_sql}) AS src
    """)

validation_sql = f"""
COPY (
  {' UNION ALL '.join(checks)}
) TO '{validation_summary_path}'
(HEADER, DELIMITER ',');
"""
con.execute(validation_sql)

summary_df = con.sql(f"SELECT * FROM read_csv('{validation_summary_path}', header = true)").df()
print(f"Validation summary → {DELIVERY_VALIDATION_DIR / 'summary.csv'}")
display(summary_df)

if "status" in summary_df.columns:
    all_pass = (summary_df["status"] == "PASS").all()
    print(f"\nValidation checks: {'ALL PASS' if all_pass else 'FAILURES DETECTED — review before publishing'}")

## 11. Create manifest

Index every exported file with format, source table, and row count. See `docs/10_export/delivery_package.md`.

In [ ]:
manifest_entries = []

if EXPORT_PARQUET:
    manifest_entries.append({
        "file_name": PARQUET_OUTPUT_PATH.name,
        "relative_path": f"data/{PARQUET_OUTPUT_PATH.name}",
        "format": "parquet",
        "source_table": SOURCE_TABLE_NAME,
        "row_count": int(export_row_count),
    })

if EXPORT_CSV:
    manifest_entries.append({
        "file_name": CSV_OUTPUT_PATH.name,
        "relative_path": f"data/{CSV_OUTPUT_PATH.name}",
        "format": "csv",
        "source_table": SOURCE_TABLE_NAME,
        "row_count": int(export_row_count),
    })

if EXPORT_GEOPARQUET and GEOMETRY_COLUMN:
    manifest_entries.append({
        "file_name": GEOPARQUET_OUTPUT_PATH.name,
        "relative_path": f"data/{GEOPARQUET_OUTPUT_PATH.name}",
        "format": "geoparquet",
        "source_table": SOURCE_TABLE_NAME,
        "row_count": int(export_row_count),
    })

if EXPORT_PARTITIONED_PARQUET and PARTITION_COLUMNS:
    manifest_entries.append({
        "file_name": f"{PARTITIONED_PARQUET_DIR.name}/",
        "relative_path": f"data/{PARTITIONED_PARQUET_DIR.name}/",
        "format": "partitioned_parquet",
        "source_table": SOURCE_TABLE_NAME,
        "row_count": int(export_row_count),
    })

manifest_df = pd.DataFrame(manifest_entries)
manifest_df.to_csv(DELIVERY_MANIFEST_PATH, index=False)

print(f"Manifest → {DELIVERY_MANIFEST_PATH}")
display(manifest_df)

## 12. Create delivery README

Write a human-readable handoff note at the package root.

In [ ]:
if COPY_EXPORTS_TO_PACKAGE:
    if EXPORT_PARQUET and PARQUET_OUTPUT_PATH.exists():
        shutil.copy2(PARQUET_OUTPUT_PATH, DELIVERY_DATA_DIR / PARQUET_OUTPUT_PATH.name)
    if EXPORT_CSV and CSV_OUTPUT_PATH.exists():
        shutil.copy2(CSV_OUTPUT_PATH, DELIVERY_DATA_DIR / CSV_OUTPUT_PATH.name)
    if EXPORT_GEOPARQUET and GEOMETRY_COLUMN and GEOPARQUET_OUTPUT_PATH.exists():
        shutil.copy2(GEOPARQUET_OUTPUT_PATH, DELIVERY_DATA_DIR / GEOPARQUET_OUTPUT_PATH.name)
    if EXPORT_PARTITIONED_PARQUET and PARTITION_COLUMNS and PARTITIONED_PARQUET_DIR.exists():
        dest_part = DELIVERY_DATA_DIR / PARTITIONED_PARQUET_DIR.name
        if dest_part.exists():
            shutil.rmtree(dest_part)
        shutil.copytree(PARTITIONED_PARQUET_DIR, dest_part)
    print(f"Copied exports into {DELIVERY_DATA_DIR}")

file_lines = []
for entry in manifest_entries:
    file_lines.append(f"- `{entry['relative_path']}` — {entry['format']} ({entry['row_count']:,} rows)")
files_section = "\n".join(file_lines) if file_lines else "- _(no files exported)_"

readme_text = f"""# Delivery Package — {DELIVERY_DATE}

## Contents
{files_section}
- `validation/summary.csv` — row counts and measure checks
- `manifest.csv` — machine-readable file index

## Source
Built from `{SOURCE_TABLE_NAME}` via DuckDB workflow template EXPORT-BASE.

## Open instructions
- **Parquet**: DuckDB, pandas (`pd.read_parquet`), Polars
- **CSV**: Excel (Data → From Text/CSV) or any spreadsheet tool
- **GeoParquet**: DuckDB with `spatial`, GeoPandas (`gpd.read_parquet`), QGIS

## Validation
Checks in `validation/summary.csv` were generated at export time. Confirm all `status` values are `PASS` before publishing.

## Package structure
```text
output/
├── data/
├── validation/
├── README.md
└── manifest.csv
```
"""

DELIVERY_README_PATH.write_text(readme_text)
print(f"README → {DELIVERY_README_PATH}")
print("\n--- README preview ---")
print(readme_text)

## 13. Final delivery checklist

Verify the package before handoff. All assertions should pass; investigate any failures before publishing.

### Automated checks

In [ ]:
checklist = []

# 1. Package folders exist
for folder in (DELIVERY_DATA_DIR, DELIVERY_VALIDATION_DIR):
    ok = folder.is_dir()
    checklist.append((f"Folder exists: {folder.name}/", ok))

# 2. README and manifest exist
checklist.append(("README.md exists", DELIVERY_README_PATH.is_file()))
checklist.append(("manifest.csv exists", DELIVERY_MANIFEST_PATH.is_file()))

# 3. Every manifest file exists on disk
for _, row in manifest_df.iterrows():
    rel = row["relative_path"].rstrip("/")
    file_path = DELIVERY_PACKAGE_DIR / rel
    if str(row["relative_path"]).endswith("/"):
        checklist.append((f"Partition dir exists: {rel}", file_path.is_dir()))
    else:
        checklist.append((f"File exists: {rel}", file_path.is_file()))

# 4. Manifest row counts match file reads
if EXPORT_PARQUET and (DELIVERY_DATA_DIR / PARQUET_OUTPUT_PATH.name).exists():
    pkg_parquet = (DELIVERY_DATA_DIR / PARQUET_OUTPUT_PATH.name).resolve().as_posix()
    n = con.sql(f"SELECT COUNT(*) AS n FROM read_parquet('{pkg_parquet}')").df().n.iloc[0]
    expected = manifest_df.loc[manifest_df["format"] == "parquet", "row_count"].iloc[0]
    checklist.append(("Parquet row count matches manifest", n == expected))

if EXPORT_CSV and (DELIVERY_DATA_DIR / CSV_OUTPUT_PATH.name).exists():
    pkg_csv = (DELIVERY_DATA_DIR / CSV_OUTPUT_PATH.name).resolve().as_posix()
    n = con.sql(f"SELECT COUNT(*) AS n FROM read_csv('{pkg_csv}', header = true)").df().n.iloc[0]
    expected = manifest_df.loc[manifest_df["format"] == "csv", "row_count"].iloc[0]
    checklist.append(("CSV row count matches manifest", n == expected))

# 5. Validation summary all PASS
if "status" in summary_df.columns:
    checklist.append(("All validation checks PASS", (summary_df["status"] == "PASS").all()))

checklist_df = pd.DataFrame(checklist, columns=["check", "passed"])
display(checklist_df)

all_ok = checklist_df["passed"].all()
print(f"\nDELIVERY READY: {'YES' if all_ok else 'NO — fix failures above'}")

print("\n--- Package inventory ---")
for p in sorted(DELIVERY_PACKAGE_DIR.rglob("*")):
    if p.is_file():
        print(f"  {p.relative_to(DELIVERY_PACKAGE_DIR)}  ({p.stat().st_size:,} bytes)")

### Manual checklist

- [ ] **Validation gate** — `notebooks/03_validation_base.ipynb` passed before export
- [ ] **Recipient** — README names the audience and any usage constraints
- [ ] **Spatial** — GeoParquet CRS and geometry validity confirmed for GIS consumers
- [ ] **Partition layout** — partition folders match consumer expectations (`docs/10_export/partitioned_parquet_export.md`)
- [ ] **Versioning** — use a new `delivery_{date}` folder per publish; never overwrite a signed-off package
- [ ] **Transfer** — zip or upload package for SFTP / object storage (`docs/10_export/delivery_package.md`)

### Assumptions (example — replace for your project)

- **Source:** TPC-H `lineitem` Parquet from `shell.duckdb.org` stands in for `curated.fct_orders`; swap `SOURCE_TABLE_NAME` for your pipeline.
- **Formats:** Parquet is the primary analytics format; CSV is a spreadsheet derivative from the same export query.
- **Partitions:** `order_month` demonstrates hive-style layout; clear `PARTITION_COLUMNS` for small tables.
- **Spatial:** Not used in this example; set `GEOMETRY_COLUMN = 'geom'` after spatial ingest (`docs/03_spatial_ingestion/`).

---

_Close the DuckDB connection when finished, or leave it open for the next notebook in the session._

In [ ]:
# con.close()
# print("Connection closed.")